In [1]:
# Import TensorFlow, a library used for building and training
# machine learning and deep learning models.
import tensorflow as tf

# Import Sequential, which allows building a neural network
# layer by layer in a linear stack.
from tensorflow.keras.models import Sequential

# Import core CNN layers.
# Conv2D extracts image features using filters.
# MaxPooling2D reduces spatial size.
# Flatten converts 2D data to 1D.
# Dense creates fully connected layers.
# Dropout randomly disables neurons to prevent overfitting.
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Import ImageDataGenerator, which loads images from folders
# and applies preprocessing and augmentation.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Import Adam optimizer, which updates model weights during
# training using adaptive learning rates.
from tensorflow.keras.optimizers import Adam

I0000 00:00:1777171506.726208   44813 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777171506.877197   44813 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777171511.902778   44813 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Define TRAIN_DIR as the relative path to the training data.
# The path "./../data/train" means go up one directory and
# then into the "data/train" folder.
TRAIN_DIR = "./../data/train"

# Define VAL_DIR as the relative path to validation data.
# This folder should contain unseen images for evaluation.
VAL_DIR = "./../data/validation"

# Define IMG_SIZE as a tuple specifying the target width
# and height (128 pixels by 128 pixels) for resizing images.
IMG_SIZE = (128, 128)

# Define BATCH_SIZE as the number of images processed in
# one forward/backward pass during training.
BATCH_SIZE = 32

In [3]:
# Create train_datagen, an ImageDataGenerator instance.
# rescale=1./255 normalizes pixel values from [0,255]
# to [0,1].
# rotation_range=20 randomly rotates images up to 20°.
# zoom_range=0.2 randomly zooms images by up to 20%.
# horizontal_flip=True randomly flips images horizontally.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Create train_generator using flow_from_directory.
# TRAIN_DIR is the directory containing class folders.
# target_size=IMG_SIZE resizes images to 128x128.
# batch_size=BATCH_SIZE defines images per batch.
# class_mode='binary' assigns labels 0/1 for two classes.
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Create val_datagen for validation data.
# Only rescale is applied to keep validation data unchanged.
val_datagen = ImageDataGenerator(
    rescale=1./255
)

# Create val_generator using flow_from_directory.
# VAL_DIR is the directory with validation images.
# target_size=IMG_SIZE resizes images to 128x128.
# batch_size=BATCH_SIZE defines images per batch.
# class_mode='binary' ensures binary labels.
val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 1998 images belonging to 2 classes.
Found 501 images belonging to 2 classes.


In [4]:
# Create a Sequential model, which stacks layers in order.
model = Sequential()

# Add a Conv2D layer.
# 32 is the number of filters (feature detectors).
# (3,3) is the kernel size.
# activation='relu' introduces non-linearity.
# input_shape=(128,128,3) defines image dimensions and RGB.
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))

# Add MaxPooling2D layer.
# (2,2) reduces spatial dimensions by taking max values.
model.add(MaxPooling2D(2, 2))

# Add another Conv2D layer with 64 filters.
# This captures more complex patterns.
model.add(Conv2D(64, (3, 3), activation='relu'))

# Add another MaxPooling2D layer to reduce size further.
model.add(MaxPooling2D(2, 2))

# Add a deeper Conv2D layer with 128 filters.
# This learns higher-level features.
model.add(Conv2D(128, (3, 3), activation='relu'))

# Add MaxPooling2D again to reduce feature map size.
model.add(MaxPooling2D(2, 2))

# Flatten the 2D feature maps into a 1D vector.
# This prepares data for fully connected layers.
model.add(Flatten())

# Add a Dense layer with 128 neurons.
# activation='relu' enables learning complex relationships.
model.add(Dense(128, activation='relu'))

# Add Dropout layer with rate 0.5.
# This randomly disables 50% of neurons during training.
model.add(Dropout(0.5))

# Add output Dense layer.
# 1 neuron outputs probability for binary classification.
# activation='sigmoid' maps output to range [0,1].
model.add(Dense(1, activation='sigmoid'))

/home/deniz/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777171543.047904   44813 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
# Compile the model.

# optimizer=Adam(...) defines weight update strategy.
# learning_rate=0.0001 controls step size of updates.
# loss='binary_crossentropy' measures classification error.
# metrics=['accuracy'] tracks accuracy during training.
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train the model using model.fit.
# train_generator supplies training data batches.
# validation_data=val_generator evaluates performance.
# epochs=10 defines number of full training passes.
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10


I0000 00:00:1777171563.444211   44813 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


63/63 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.7082 - loss: 0.5729 - val_accuracy: 0.7226 - val_loss: 0.6100
Epoch 2/10
39/63 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.7601 - loss: 0.5092

In [ ]:
# Import matplotlib for plotting training results.
import matplotlib.pyplot as plt

# Plot training accuracy over epochs.
# history.history['accuracy'] contains training accuracy.
plt.plot(history.history['accuracy'], label='train')

# Plot validation accuracy over epochs.
# history.history['val_accuracy'] contains validation data.
plt.plot(history.history['val_accuracy'], label='validation')

# Add legend to distinguish lines.
plt.legend()

# Set plot title.
plt.title("Accuracy")

# Display the plot.
plt.show()

In [ ]:
# Import image preprocessing utilities.
from tensorflow.keras.preprocessing import image

# Import NumPy for numerical operations.
import numpy as np

# Define path to test image.
img_path = "./../test_images/test_image.jpg"

# Load image from path.
# target_size=IMG_SIZE resizes image to 128x128.
img = image.load_img(img_path, target_size=IMG_SIZE)

# Convert image to NumPy array and normalize pixel values.
img_array = image.img_to_array(img) / 255.0

# Expand dimensions to match model input shape.
# axis=0 adds batch dimension (1, height, width, channels).
img_array = np.expand_dims(img_array, axis=0)

# Predict class probability using trained model.
prediction = model.predict(img_array)

# Interpret prediction result.
# prediction[0][0] accesses probability value.
# Threshold 0.5 determines class label.
if prediction[0][0] > 0.5:
    print("Malignant (cancerous cells).")
else:
    print("Benign (non-cancerous tumour).")